In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install faiss-cpu transformers sentencepiece pymilvus -q

In [25]:
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np

class EmbeddingModel:
    def __init__(self, model_name="BAAI/bge-m3"):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(self.device)

    @torch.no_grad()
    def encode(self, texts):
        inputs = self.tokenizer(
            texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=256
        ).to(self.device)

        outputs = self.model(**inputs)
        dense = outputs.last_hidden_state[:, 0]

        dense = dense / dense.norm(dim=1, keepdim=True)
        return dense.cpu().numpy()


In [26]:
# FAISS 벡터스토어

import faiss
import numpy as np

class FaissVectorStore:
    def __init__(self, dim, index_path="/content/drive/MyDrive/rag-mmlu-ewha/data/ewha_index.faiss"):
        self.index_path = index_path
        self.index = faiss.IndexFlatIP(dim)  # cosine similarity

    def build_index(self, vectors):
        self.index.add(vectors)

    def save(self):
        faiss.write_index(self.index, self.index_path)

    def load(self):
        self.index = faiss.read_index(self.index_path)

    def search(self, q_emb, top_k=5):
        D, I = self.index.search(q_emb, top_k)
        return D[0], I[0]


In [27]:
# Ewha corpus 읽기

import json

corpus_path = "/content/drive/MyDrive/rag-mmlu-ewha/data/ewha_corpus.jsonl"  # 업로드해서 경로 확인

corpus_texts = []
with open(corpus_path, "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        corpus_texts.append(obj["text"])

len(corpus_texts)


26

In [28]:
# 임베딩 생성
model = EmbeddingModel()

print("Encoding Ewha corpus...")
dense_vectors = model.encode(corpus_texts)
dense_vectors.shape

# 인덱스 생성
index_path = "/content/drive/MyDrive/rag-mmlu-ewha/data/ewha_index.faiss"

store = FaissVectorStore(dim=dense_vectors.shape[1], index_path=index_path)
store.build_index(dense_vectors)
store.save()

print("Saved index →", index_path)

Encoding Ewha corpus...
Saved index → /content/drive/MyDrive/rag-mmlu-ewha/data/ewha_index.faiss


In [29]:
import pandas as pd

test_df = pd.read_csv("/content/drive/MyDrive/rag-mmlu-ewha/data/testset.csv")  # 파일 경로 맞춰줘야함
test_df.head()

,prompts,answers
0,QUESTION1) 재학 중인 학생이 휴학을 하려면 학기 개시일로부터 며칠 이내에 ...,(D)
1,QUESTION2) '재입학은 a회에 한하여 할 수 있다. 다만 제 28조제4호에 ...,(A)
2,QUESTION3) 학생이 소속 학과 또는 전공 이외의 전공 교과목을 총장이 정하는...,(C)
3,QUESTION4) 다음 보기의 학생들 중 제적을 당하지 않는 사람을 고르면?\n(...,(D)
4,QUESTION5) 2019학년도 휴먼기계바이오공학부의 입학 정원은 몇 명인가? \...,(C)


In [30]:
q1 = test_df.iloc[0]["prompts"]
print(q1)

# BGE-M3 임베딩 → 검색
query = q1
q_emb = model.encode([query])  # BGE-M3 encode
scores, idxs = store.search(q_emb, top_k=5)
print(scores)
print(idxs)

print("=== TOP 5 검색 결과 ===")
for rank, i in enumerate(idxs):
    print(f"[{rank+1}] score={scores[rank]:.4f}")
    print(corpus_texts[i][:500], "...")  # 앞 500자만 보기
    print("-"*80)

QUESTION1) 재학 중인 학생이 휴학을 하려면 학기 개시일로부터 며칠 이내에 휴학을 신청하야하나요?
(A) 30일
(B) 45일 
(C) 60일
(D) 90일
[0.77615154 0.6924661  0.6452466  0.6172712  0.60509884]
[ 8  4  5  6 12]
=== TOP 5 검색 결과 ===
[1] score=0.7762
제8장 휴학, 복학, 제적, 자퇴 및 재입학 (개정 2017.8.16.)

제26조(휴학) ① 질병 기타 부득이한 사정으로 3주일 이상 수강할 수 없는 자는 총장의 허가를 얻어 휴학할 수 있다. ② 총장은 건강상의 이유로 정상적인 수업을 받을 수 없다고 인정되는 자에 대하여 휴학을 명할 수 있다. (개정 1988.7.28) ③ 1회의 휴학기간은 1년 이내로 한다. 다만, 교과과정상의 필요에 따라 총장이 지정하는 학부, 학과 또는 전공에 있어서는 이를 1년으로 한다. (개정 1996.2.15) ④ 휴학기간은 통산하여 3년(건축학전공의 경우 4년, 의예과의 경우 3학기)을 초과할 수 없다. 다만, 임신, 출산 및 육아로 인한 휴학, 창업으로 인한 휴학은 2년 이내의 기간을, 군복무로 인한 휴학은 의무복무기간을 추가 휴학기간으로 허가할 수 있다. (개정 2013.2.25., 2015.9.18., 2016.2.16.) ⑤ 삭제 (1985.9.9.) ⑥ 재학 중인 자가 휴학을 하고자 하는 경 ...
--------------------------------------------------------------------------------
[2] score=0.6925
제4장 학년, 학기, 수업일수, 휴업일 및 교원의 교수시간 (개정 1998.6.23)

제10조(학년, 학기) ① 학년은 3월 1일부터 다음해 2월 말일까지로 하고 이를 제1학기와 제2학기로 나눈다. ② 제1항의 일반학기 외에 하기방학과 동기방학중에 총장이 정하는 바에 따라 각각 계절학기를 둘 수 있다.

제11조(수업일수) ① 수업일수는 매 학